# Modelos lineales generalizados: abandono de clientes y tipos de vino

**Autora:** Silvia Katerin Angel Camargo

# Origen de los datasets

**Ejercicio 1 (abandono de clientes).** Se usó la base *Bank Customer Churn Prediction* (archivo `Churn_Modelling.csv`), obtenida de Kaggle. Contiene 10000 clientes de un banco, de Francia, Alemania y España, con variables financieras y sociodemográficas y una variable que indica si el cliente abandonó la entidad.

**Ejercicio 2 (tipos de vino).** Se usó la base *Wine* (archivo `wine.data`) del UCI Machine Learning Repository (Aeberhard, Coomans y de Vel, 1991), disponible en https://archive.ics.uci.edu/dataset/109/wine. Reúne los resultados de análisis químicos de 178 vinos cultivados en una misma región de Italia, provenientes de tres cultivares distintos, con 13 variables químicas.

**Reproducibilidad.** Los archivos se leen con rutas relativas, por lo que deben estar en la misma carpeta que este cuaderno (si `wine.data` no está, se descarga de UCI). Las dependencias están en `requirements.txt` (`pip install -r requirements.txt`) y las particiones de entrenamiento y prueba usan una semilla fija.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
SEED = 2026                      # reproducibilidad de las particiones
sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 4)


def md(texto):
    """Muestra texto en formato Markdown (para interpretaciones con cifras reales)."""
    display(Markdown(texto))


def efecto(o):
    """Cambio porcentual en los momios a partir de un odds ratio."""
    if o >= 1:
        return f"aumentan en {(o - 1) * 100:.1f} %"
    return f"disminuyen en {(1 - o) * 100:.1f} %"


def predecir_multi(modelo, datos):
    """Probabilidades de las clases 1, 2 y 3 como DataFrame."""
    p = pd.DataFrame(np.asarray(modelo.predict(datos)), index=datos.index)
    p.columns = [1, 2, 3]
    return p


def fp(p):
    """Formato de p-valor."""
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

# Ejercicio 1. Regresión logística binaria: abandono de clientes

**Situación problema:** Una entidad bancaria desea identificar factores asociados al abandono de clientes. Se construye un **MLG logístico binario** para estimar la probabilidad de abandono a partir de características financieras y sociodemográficas.

## 1.1. Carga y revisión de datos

In [ ]:
# El archivo debe estar en la misma carpeta que este cuaderno
archivo_churn = "Churn_Modelling.csv"
if not os.path.exists(archivo_churn):
    raise FileNotFoundError("No se encuentra Churn_Modelling.csv en la carpeta del cuaderno.")

clientes = pd.read_csv(archivo_churn)

# Nombres en minúscula para evitar errores de escritura
clientes.columns = clientes.columns.str.lower()

# Estructura, dimensiones y valores faltantes
clientes.info()
print(clientes.shape)
print(clientes.isna().sum())

In [ ]:
# Preparación: nombres más claros y variables indicadoras
clientes = clientes.rename(columns={
    "creditscore": "credit_score", "geography": "country",
    "numofproducts": "products_number", "hascrcard": "credit_card",
    "isactivemember": "active_member", "estimatedsalary": "estimated_salary",
    "exited": "churn",
})

# Se retiran los identificadores porque no son predictores
clientes = clientes.drop(columns=[c for c in ["customer_id", "customerid", "rownumber", "id", "surname"]
                                  if c in clientes.columns])

clientes["churn"] = clientes["churn"].astype(int)          # 1 = abandono, 0 = permanece
clientes["balance_miles"] = clientes["balance"] / 1000     # saldo en miles

# Categorías de referencia: France (país) y Male (género)
clientes["country_Germany"] = (clientes["country"] == "Germany").astype(int)
clientes["country_Spain"] = (clientes["country"] == "Spain").astype(int)
clientes["gender_Female"] = (clientes["gender"] == "Female").astype(int)

md(f"**Descripción del conjunto de datos:** La base contiene {len(clientes)} registros y 11 variables originales "
   f"(después de retirar los identificadores: número de fila, id y apellido del cliente): puntaje de crédito, país, "
   f"género, edad, antigüedad, saldo, número de productos, tenencia de tarjeta de crédito, cliente activo, salario "
   f"estimado y la variable de abandono. El total de valores faltantes es {int(clientes.isna().sum().sum())}.")

## 1.2. Variables y análisis exploratorio

**Variable respuesta:** `churn`, con 1 = el cliente abandonó el banco y 0 = el cliente permanece.\
**Variables predictoras seleccionadas:** `credit_score` (la solvencia puede influir en la permanencia), `country` y `gender` (diferencias sociodemográficas), `age` (el ciclo de vida del cliente), `tenure` (antigüedad como vínculo con el banco), `balance_miles` (nivel de recursos en el banco), `products_number` (profundidad de la relación) y `active_member` (nivel de uso). Se excluyen `credit_card` y `estimated_salary` por aportar poca información adicional en un primer modelo y para mantenerlo parsimonioso.

In [ ]:
# Tabla de frecuencias de la variable respuesta
tab_churn = clientes["churn"].value_counts().sort_index()
display(pd.DataFrame({"Clientes": tab_churn, "Porcentaje": (100 * tab_churn / tab_churn.sum()).round(1)}))

# Tasas de abandono por grupo (para la interpretación)
tasa = clientes["churn"].mean()
tasa_pais = clientes.groupby("country")["churn"].mean()
tasa_act = clientes.groupby("active_member")["churn"].mean()
edad_med = clientes.groupby("churn")["age"].mean()

# Resumen descriptivo de las variables elegidas
clientes[["credit_score", "age", "tenure", "balance_miles", "products_number"]].describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.4))
sns.countplot(data=clientes, x="churn", color="steelblue", ax=ax)
ax.set(xlabel="Abandono (0 = no, 1 = sí)", ylabel="Número de clientes",
       title="Figura 1. Distribución del abandono de clientes")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.8))
sns.boxplot(data=clientes, x="churn", y="age", color="lightblue", ax=ax)
ax.set(xlabel="Abandono (0 = no, 1 = sí)", ylabel="Edad (años)",
       title="Figura 2. Edad de los clientes según abandono")
plt.tight_layout(); plt.show()

In [ ]:
tasa_pa = clientes.groupby(["country", "active_member"])["churn"].mean().reset_index()
fig, ax = plt.subplots(figsize=(6, 3.8))
sns.barplot(data=tasa_pa, x="country", y="churn", hue="active_member", ax=ax)
ax.set(xlabel="País", ylabel="Tasa de abandono",
       title="Figura 3. Tasa de abandono por país y condición de cliente activo")
ax.legend(title="Cliente activo")
plt.tight_layout(); plt.show()

In [ ]:
paises = "; ".join(f"{k} {100 * v:.1f} %" for k, v in tasa_pais.items())
md(f"**Interpretación del análisis exploratorio:** La tasa global de abandono es de {100 * tasa:.1f} %, por lo que la "
   f"clase de interés es minoritaria (base desbalanceada). La edad promedio de quienes abandonan es {edad_med[1]:.1f} "
   f"años, frente a {edad_med[0]:.1f} años de quienes permanecen, lo que sugiere que la edad se asocia con el abandono. "
   f"La tasa de abandono de los clientes no activos es {100 * tasa_act[0]:.1f} %, frente a {100 * tasa_act[1]:.1f} % de "
   f"los activos. Por país, las tasas son {paises}.")

## 1.3. Formulación y ajuste del modelo

**Pregunta de análisis:** ¿Cómo se relacionan la edad, el saldo, la actividad, el país, el género y la relación con el banco (productos y antigüedad) con la probabilidad de que un cliente abandone?\
**Modelo y enlace:** Familia binomial; enlace logit.\
**Categoría de referencia / evento de interés:** El evento codificado como 1 es el **abandono**. Las categorías de referencia de los factores son `France` (país) y `Male` (género).

In [ ]:
# Partición entrenamiento (80 %) y prueba (20 %)
train, test = train_test_split(clientes, test_size=0.2, random_state=SEED)

# Ajuste del modelo con los datos de entrenamiento
formula_bin = ("churn ~ credit_score + country_Germany + country_Spain + gender_Female + age + tenure "
               "+ balance_miles + products_number + active_member")
modelo_binario = smf.glm(formula_bin, data=train, family=sm.families.Binomial()).fit()
print(modelo_binario.summary())

In [ ]:
# Razones de momios (odds ratios) con IC de Wald al 95 %
ic = modelo_binario.conf_int()
tab_or = pd.DataFrame({
    "Coeficiente": modelo_binario.params,
    "OR": np.exp(modelo_binario.params),
    "IC_inf": np.exp(ic[0]),
    "IC_sup": np.exp(ic[1]),
    "p_valor": modelo_binario.pvalues,
}).round(4)
display(tab_or)

OR = np.exp(modelo_binario.params)
P = modelo_binario.pvalues

In [ ]:
md(f"""**Interpretación de los coeficientes:** Un odds ratio (OR) compara momios, no probabilidades.

- *Edad:* el coeficiente es {modelo_binario.params['age']:.3f} (p {fp(P['age'])}) y el OR es {OR['age']:.3f}. Por cada año adicional de edad, y manteniendo constantes las demás variables, los momios de abandono {efecto(OR['age'])}.
- *Cliente activo:* el OR es {OR['active_member']:.3f} (p {fp(P['active_member'])}). Frente a un cliente no activo, los momios de abandono de un cliente activo {efecto(OR['active_member'])}.
- *País (Germany vs. France):* el OR es {OR['country_Germany']:.3f} (p {fp(P['country_Germany'])}); los momios de abandono de los clientes de Alemania {efecto(OR['country_Germany'])} respecto a los de Francia.
- *Género (Female vs. Male):* el OR es {OR['gender_Female']:.3f} (p {fp(P['gender_Female'])}); los momios de las mujeres {efecto(OR['gender_Female'])} respecto a los hombres.
- *Saldo (por cada 1000 unidades):* el OR es {OR['balance_miles']:.3f} (p {fp(P['balance_miles'])}); los momios {efecto(OR['balance_miles'])} por cada 1000 unidades adicionales de saldo.

Un OR mayor que 1 (coeficiente positivo) indica mayor momio de abandono; un OR menor que 1 (coeficiente negativo), menor momio. Una variable con p mayor a 0.05 no presenta evidencia suficiente de asociación, ajustando por las demás.""")

## 1.4. Probabilidades estimadas, matriz de confusión y exactitud (datos de prueba)

In [ ]:
# Probabilidades estimadas para los clientes de prueba
prob_test = pd.Series(np.asarray(modelo_binario.predict(test)), index=test.index)
display(prob_test.head().round(3))


def evaluar(real, prob, umbral):
    """Matriz de confusión, exactitud, sensibilidad y especificidad para un umbral."""
    pred = (prob >= umbral).astype(int)
    cm = pd.crosstab(real, pred).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    cm.index.name, cm.columns.name = "Real", "Predicho"
    exactitud = (cm.loc[0, 0] + cm.loc[1, 1]) / cm.values.sum()
    sensibilidad = cm.loc[1, 1] / cm.loc[1].sum()
    especificidad = cm.loc[0, 0] / cm.loc[0].sum()
    return cm, exactitud, sensibilidad, especificidad


# Clasificación con umbral 0.50
cm, exactitud, sensibilidad, especificidad = evaluar(test["churn"], prob_test, 0.50)
print("Matriz de confusión en prueba (umbral 0.50)")
display(cm)

# Exactitud de un clasificador que siempre predice "no abandona"
base_mayor = test["churn"].value_counts(normalize=True).max()

# Segundo umbral: igual a la tasa de abandono (clase minoritaria)
umbral2 = train["churn"].mean()
cm2, exactitud2, sensibilidad2, especificidad2 = evaluar(test["churn"], prob_test, umbral2)
print("Matriz de confusión en prueba (umbral = tasa de abandono)")
display(cm2)

In [ ]:
pm = prob_test.groupby(test["churn"]).mean()
real = test["churn"].map({0: "Permanece", 1: "Abandona"})

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharex=True)
for ax, (etiqueta, color) in zip(axes, [("Permanece", "steelblue"), ("Abandona", "indianred")]):
    ax.hist(prob_test[real == etiqueta], bins=30, color=color, edgecolor="white")
    ax.axvline(0.50, linestyle="--", color="black")
    ax.axvline(umbral2, linestyle=":", color="black")
    ax.set(title=etiqueta, xlabel="Probabilidad estimada de abandono", ylabel="Número de clientes")
fig.suptitle("Probabilidades estimadas en prueba (discontinua: umbral 0.50; punteada: umbral = tasa de abandono)",
             fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
nuevo = pd.DataFrame([(a, act) for act in (0, 1) for a in range(18, 91)], columns=["age", "active_member"])
for col in ["credit_score", "tenure", "balance_miles", "products_number"]:
    nuevo[col] = train[col].mean()
nuevo["country_Germany"] = 0       # cliente de Francia
nuevo["country_Spain"] = 0
nuevo["gender_Female"] = 0         # hombre
nuevo["prob"] = np.asarray(modelo_binario.predict(nuevo))

fig, ax = plt.subplots(figsize=(6, 3.8))
for act, etiqueta in [(0, "No"), (1, "Sí")]:
    sub = nuevo[nuevo["active_member"] == act]
    ax.plot(sub["age"], sub["prob"], label=etiqueta, linewidth=2)
ax.set(xlabel="Edad (años)", ylabel="Probabilidad estimada de abandono",
       title="Probabilidad estimada según edad y condición de cliente activo")
ax.legend(title="Cliente activo")
plt.tight_layout(); plt.show()


def p_edad(edad, act):
    return nuevo.loc[(nuevo["age"] == edad) & (nuevo["active_member"] == act), "prob"].iloc[0]

In [ ]:
md(f"**Lectura de los gráficos:** En los clientes de prueba que sí abandonaron, la probabilidad estimada promedio fue "
   f"{pm[1]:.3f}, frente a {pm[0]:.3f} en los que permanecieron, es decir, el modelo asigna probabilidades más altas a "
   f"quienes se van. Sin embargo, las dos distribuciones se solapan: {100 * (1 - sensibilidad):.1f} % de los abandonos "
   f"tiene una probabilidad estimada menor a 0.50, lo que explica la baja sensibilidad con ese umbral. En la curva, para "
   f"un cliente de Francia, hombre y con las demás variables en su promedio, la probabilidad estimada a los 30 años es "
   f"{p_edad(30, 0):.3f} si no es activo y {p_edad(30, 1):.3f} si es activo; a los 60 años pasa a {p_edad(60, 0):.3f} y "
   f"{p_edad(60, 1):.3f}, respectivamente. La probabilidad crece con la edad en ambos grupos y los clientes activos "
   f"tienen siempre una probabilidad menor.")

md(f"**Interpretación:** Cada probabilidad estimada es la probabilidad que el modelo asigna a que ese cliente abandone, "
   f"dadas sus características. Con el umbral de 0.50, el modelo obtiene en los {len(test)} clientes de prueba una "
   f"exactitud de {100 * exactitud:.1f} %, con sensibilidad (abandonos detectados) de {100 * sensibilidad:.1f} % y "
   f"especificidad de {100 * especificidad:.1f} %. Como referencia, un clasificador que prediga siempre "
   f"\"no abandona\" alcanzaría {100 * base_mayor:.1f} %, de modo que la exactitud por sí sola es engañosa cuando la "
   f"clase es desbalanceada. Al bajar el umbral a {umbral2:.2f}, la sensibilidad pasa a {100 * sensibilidad2:.1f} % y la "
   f"especificidad a {100 * especificidad2:.1f} % (exactitud {100 * exactitud2:.1f} %): se detectan más abandonos a costa "
   f"de más falsas alarmas. Estos resultados se calcularon con datos que **no** participaron en el ajuste.")

## 1.5. Conclusión del ejercicio 1

In [ ]:
md(f"El modelo logístico binario permite concluir que el abandono se asocia principalmente con la edad (a mayor edad, "
   f"los momios de abandono {efecto(OR['age'])} por año), con la condición de cliente activo (los activos "
   f"{efecto(OR['active_member'])}) y con el país de residencia, además del género (las mujeres presentan momios "
   f"mayores) y del saldo. En cambio, el puntaje de crédito tiene un efecto pequeño, y la antigüedad, el número de "
   f"productos y residir en España (frente a Francia) no muestran evidencia de asociación (p > 0.05). En datos de prueba "
   f"la exactitud es de {100 * exactitud:.1f} %, pero la sensibilidad es de {100 * sensibilidad:.1f} %, lo que refleja "
   f"la principal limitación: al ser el abandono un evento minoritario, el umbral de 0.50 deja sin detectar a una parte "
   f"importante de los clientes que se van. Además, el estudio es observacional, por lo que las asociaciones no son "
   f"causales, y se asumió una relación lineal en el logit para las variables numéricas.")

# Ejercicio 2. Regresión logística multinomial: tipos de vino

**Situación problema:** Una empresa desea analizar cómo las características químicas permiten diferenciar tres clases de vino. Se construye un **modelo logístico multinomial** para estimar la probabilidad de pertenencia a cada clase.

## 2.1. Carga y revisión de datos

In [ ]:
# Se usa el archivo local "wine.data" si existe; si no, se lee desde UCI.
url_vino = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data"
fuente_vino = "wine.data" if os.path.exists("wine.data") else url_vino

columnas_vino = [
    "clase", "Alcohol", "Malicacid", "Ash", "Alcalinity_of_ash",
    "Magnesium", "Total_phenols", "Flavanoids", "Nonflavanoid_phenols",
    "Proanthocyanins", "Color_intensity", "Hue", "OD280_OD315", "Proline",
]
vinos = pd.read_csv(fuente_vino, header=None, names=columnas_vino)   # tres clases nominales: 1, 2 y 3

vinos.info()
print(vinos["clase"].value_counts().sort_index())
print(vinos.isna().sum())

In [ ]:
conteo = vinos["clase"].value_counts().sort_index()
clases_txt = "; ".join(f"{k}: {v}" for k, v in conteo.items())
md(f"**Descripción del conjunto de datos:** La base contiene {len(vinos)} vinos, clasificados en {vinos['clase'].nunique()} "
   f"clases ({clases_txt} vinos) y 13 variables químicas continuas. El total de valores faltantes es "
   f"{int(vinos.isna().sum().sum())}.")

## 2.2. Variables y análisis exploratorio

**Variable respuesta:** `clase` (clases 1, 2 y 3).\
**Variables predictoras seleccionadas:** `Alcohol`, `Malicacid` y `Color_intensity`. Son variables interpretables para una empresa vinícola y, según el análisis exploratorio, presentan diferencias visibles entre clases.\
**Categoría de referencia:** la clase 1. Cada coeficiente compara la clase 2 (o la 3) **contra la clase 1**, es decir, describe cómo cambian los momios de pertenecer a esa clase en lugar de la clase 1 cuando cambia el predictor.

In [ ]:
# Resumen descriptivo de los predictores seleccionados, por clase
res_vinos = vinos.groupby("clase")[["Alcohol", "Malicacid", "Color_intensity"]].agg(["mean", "std"]).round(2)
res_vinos.columns = ["Alcohol media", "Alcohol DE", "Ác. málico media", "Ác. málico DE", "Color media", "Color DE"]
display(res_vinos)
vinos[["Alcohol", "Malicacid", "Color_intensity"]].describe().round(3)

In [ ]:
for var, etiqueta, fig_n in [("Alcohol", "Alcohol", 4), ("Color_intensity", "Intensidad del color", 5),
                             ("Malicacid", "Ácido málico", 6)]:
    fig, ax = plt.subplots(figsize=(5, 3.6))
    sns.boxplot(data=vinos, x="clase", y=var, hue="clase", palette="Set2", legend=False, ax=ax)
    ax.set(xlabel="Clase de vino", ylabel=etiqueta, title=f"Figura {fig_n}. {etiqueta} por clase de vino")
    plt.tight_layout(); plt.show()

**Interpretación del análisis exploratorio:** Las tres clases están razonablemente balanceadas. La intensidad del color separa bien las clases: la clase 2 tiene los vinos de color menos intenso y la clase 3 los de color más intenso, con la clase 1 en una posición intermedia. El contenido de alcohol es más alto en la clase 1 y más bajo en la clase 2, con la clase 3 en medio. El ácido málico es claramente mayor en la clase 3, mientras que las clases 1 y 2 son parecidas entre sí. Los valores exactos están en la tabla de medias por clase.

## 2.3. Formulación y ajuste del modelo

**Pregunta de análisis:** ¿Cómo se relacionan el contenido de alcohol, el ácido málico y la intensidad del color con la probabilidad de que un vino pertenezca a la clase 2 o a la clase 3, en lugar de a la clase 1?\
**Modelo:** Regresión logística multinomial con categoría de referencia (clase 1).

In [ ]:
# Partición estratificada por clase: 70 % entrenamiento, 30 % prueba
train_v, test_v = train_test_split(vinos, test_size=0.3, stratify=vinos["clase"], random_state=SEED)

# Ajuste del modelo multinomial (la categoría de referencia es la clase 1, la menor)
formula_multi = "clase ~ Alcohol + Malicacid + Color_intensity"
modelo_multi = smf.mnlogit(formula_multi, data=train_v).fit(maxiter=500, disp=False)

# Coeficientes, errores estándar, z, p-valores y razones (RRR); columnas = clases 2 y 3
co = modelo_multi.params.copy();  co.columns = [2, 3]
se = modelo_multi.bse.copy();     se.columns = [2, 3]
pv = modelo_multi.pvalues.copy(); pv.columns = [2, 3]
RRR = np.exp(co)

print(modelo_multi.summary())

In [ ]:
# El intercepto no se interpreta, se omite de la tabla
tab_multi = pd.concat([
    pd.DataFrame({"Comparación": f"Clase {k} vs 1", "Variable": co.index, "Coeficiente": co[k].values,
                  "RRR": RRR[k].values, "p_valor": pv[k].values})
    for k in co.columns
], ignore_index=True)
tab_multi = tab_multi[tab_multi["Variable"] != "Intercept"]
display(tab_multi.round(4))

In [ ]:
md(f"""**Interpretación de los coeficientes:** Todas las comparaciones son respecto a la clase 1. Cada razón (exponencial del coeficiente) se interpreta como el cambio multiplicativo en los momios de pertenecer a la clase indicada, en lugar de la clase 1, por cada aumento de una unidad del predictor y manteniendo los demás constantes.

- *Color_intensity, clase 3 vs 1:* coeficiente {co.loc['Color_intensity', 3]:.3f}, razón {RRR.loc['Color_intensity', 3]:.3f} (p {fp(pv.loc['Color_intensity', 3])}). Por cada unidad adicional de intensidad del color, los momios de ser clase 3 en lugar de clase 1 {efecto(RRR.loc['Color_intensity', 3])}.
- *Color_intensity, clase 2 vs 1:* coeficiente {co.loc['Color_intensity', 2]:.3f}, razón {RRR.loc['Color_intensity', 2]:.3f} (p {fp(pv.loc['Color_intensity', 2])}). Los momios de ser clase 2 en lugar de clase 1 {efecto(RRR.loc['Color_intensity', 2])} por cada unidad adicional de color.
- *Alcohol, clase 2 vs 1:* razón {RRR.loc['Alcohol', 2]:.3f} (p {fp(pv.loc['Alcohol', 2])}). Por cada punto adicional de alcohol, los momios de ser clase 2 en lugar de clase 1 {efecto(RRR.loc['Alcohol', 2])}.
- *Alcohol, clase 3 vs 1:* razón {RRR.loc['Alcohol', 3]:.3f} (p {fp(pv.loc['Alcohol', 3])}); los momios de ser clase 3 en lugar de clase 1 {efecto(RRR.loc['Alcohol', 3])} por cada punto adicional de alcohol.
- *Malicacid, clase 3 vs 1:* razón {RRR.loc['Malicacid', 3]:.3f} (p {fp(pv.loc['Malicacid', 3])}); los momios de ser clase 3 en lugar de clase 1 {efecto(RRR.loc['Malicacid', 3])} por cada unidad adicional de ácido málico.""")

## 2.4. Probabilidades estimadas, matriz de confusión y exactitud (datos de prueba)

In [ ]:
# Probabilidades estimadas de las tres clases en los datos de prueba
prob_multi = predecir_multi(modelo_multi, test_v)
display(prob_multi.head().round(3))

# Cada fila suma 1
print(prob_multi.sum(axis=1).describe().round(4))

# Probabilidad media asignada a cada clase, según la clase real
prob_media = prob_multi.groupby(test_v["clase"].values).mean()
prob_media.index.name = "Clase real"
display(prob_media.round(3))

# Clasificación en datos de prueba: la clase de mayor probabilidad
pred_multi = prob_multi.idxmax(axis=1)
cm_multi = pd.crosstab(test_v["clase"], pred_multi).reindex(index=[1, 2, 3], columns=[1, 2, 3], fill_value=0)
cm_multi.index.name, cm_multi.columns.name = "Real", "Predicho"
display(cm_multi)

exactitud_m = np.trace(cm_multi.values) / cm_multi.values.sum()
recall_m = np.diag(cm_multi.values) / cm_multi.sum(axis=1).values   # proporción bien clasificada por clase
print(recall_m.round(3))

In [ ]:
largo = prob_multi.assign(real=test_v["clase"].values).melt(id_vars="real", var_name="clase_est", value_name="prob")
largo["clase_est"] = "Prob. clase " + largo["clase_est"].astype(str)
g = sns.catplot(data=largo, x="real", y="prob", col="clase_est", kind="box", hue="real", palette="Set2",
                legend=False, height=3.2, aspect=0.9)
g.set_axis_labels("Clase real", "Probabilidad estimada")
g.set_titles("{col_name}")
g.figure.suptitle("Probabilidades estimadas de cada clase en prueba, según la clase real", y=1.04, fontsize=10)
plt.show()

In [ ]:
nuevo_v = pd.DataFrame({
    "Color_intensity": np.linspace(vinos["Color_intensity"].min(), vinos["Color_intensity"].max(), 100),
    "Alcohol": vinos["Alcohol"].mean(),
    "Malicacid": vinos["Malicacid"].mean(),
})
pr = predecir_multi(modelo_multi, nuevo_v)

fig, ax = plt.subplots(figsize=(6, 3.8))
for k in pr.columns:
    ax.plot(nuevo_v["Color_intensity"], pr[k], label=f"Clase {k}", linewidth=2)
ax.set(xlabel="Intensidad del color", ylabel="Probabilidad estimada",
       title="Probabilidad estimada según la intensidad del color\n(alcohol y ácido málico en su promedio)")
ax.legend()
plt.tight_layout(); plt.show()

pm_v = np.diag(prob_media.values)

In [ ]:
md(f"**Lectura de los gráficos:** La probabilidad media que el modelo asigna a la clase verdadera es "
   f"{', '.join(f'{x:.3f}' for x in pm_v)} para las clases 1, 2 y 3, respectivamente (las cajas más altas en la diagonal "
   f"de la primera figura). Respecto a la intensidad del color, con alcohol y ácido málico en su promedio, la clase con "
   f"mayor probabilidad en el color más bajo observado es la {pr.iloc[0].idxmax()} y en el color más alto es la "
   f"{pr.iloc[-1].idxmax()}; en los valores intermedios las probabilidades se reparten entre clases, que es donde se "
   f"concentran los errores de clasificación.")

md(f"**Interpretación:** Para cada vino, el modelo entrega tres probabilidades (una por clase) que suman 1; la clase "
   f"predicha es la de mayor probabilidad. En los {len(test_v)} vinos de prueba, el modelo clasifica correctamente "
   f"{int(np.trace(cm_multi.values))}, para una exactitud de {100 * exactitud_m:.1f} %. La proporción bien clasificada "
   f"por clase es de {', '.join(f'{100 * x:.1f} %' for x in recall_m)} para las clases 1, 2 y 3, respectivamente. Como la "
   f"tabla de probabilidades medias muestra, el modelo asigna en promedio su mayor probabilidad a la clase verdadera. Los "
   f"resultados corresponden a datos no usados en el ajuste, pero la prueba es pequeña ({len(test_v)} vinos), por lo que "
   f"la exactitud es una estimación con bastante variabilidad.")

## 2.5. Estabilidad del modelo

La exactitud anterior depende de una sola partición con 54 vinos de prueba. Para evaluar qué tan estable es el modelo, se repitió 200 veces la partición estratificada 70 % / 30 %, ajustando el modelo multinomial en cada una y guardando la exactitud en prueba y los coeficientes.

In [ ]:
B = 200
acc_rep, filas = [], []
for b in range(B):
    tr, te = train_test_split(vinos, test_size=0.3, stratify=vinos["clase"], random_state=SEED + 1 + b)
    try:
        m_b = smf.mnlogit(formula_multi, data=tr).fit(maxiter=500, disp=False)
    except Exception:          # un ajuste que no converge se descarta
        continue
    p_b = predecir_multi(m_b, te)
    acc_rep.append((p_b.idxmax(axis=1).values == te["clase"].values).mean())
    pb = m_b.params.copy(); pb.columns = [2, 3]
    filas.append({f"{c}: {v}": pb.loc[v, c] for c in [2, 3] for v in pb.index})

acc_rep = np.array(acc_rep)
coef_rep = pd.DataFrame(filas)
print(f"Ajustes válidos: {len(acc_rep)} de {B}")

In [ ]:
# Resumen de la exactitud (%)
res_acc = pd.DataFrame({
    "Media": [acc_rep.mean()], "DE": [acc_rep.std(ddof=1)], "Mínimo": [acc_rep.min()],
    "P2.5": [np.quantile(acc_rep, 0.025)], "Mediana": [np.median(acc_rep)],
    "P97.5": [np.quantile(acc_rep, 0.975)], "Máximo": [acc_rep.max()],
}) * 100
display(res_acc.round(1))

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(100 * acc_rep, bins=15, color="steelblue", edgecolor="white")
ax.axvline(100 * acc_rep.mean(), linestyle="--", color="black")
ax.set(xlabel="Exactitud en prueba (%)", ylabel="Número de particiones",
       title="Distribución de la exactitud en 200 particiones (línea: media)")
plt.tight_layout(); plt.show()

In [ ]:
# Estabilidad de los coeficientes (sin interceptos)
cr = coef_rep.loc[:, ~coef_rep.columns.str.contains("Intercept")]
tab_est = pd.DataFrame({
    "Media": cr.mean(),
    "DE": cr.std(ddof=1),
    "RRR mediana": np.exp(cr.median()),
    "RRR P2.5": np.exp(cr.quantile(0.025)),
    "RRR P97.5": np.exp(cr.quantile(0.975)),
    "% mismo signo": 100 * cr.apply(lambda x: (np.sign(x) == np.sign(x.mean())).mean()),
})
display(tab_est.round(3))

incluyen1 = tab_est.index[(tab_est["RRR P2.5"] < 1) & (tab_est["RRR P97.5"] > 1)].tolist()
texto_inc = ", ".join(incluyen1) if incluyen1 else "ninguno"
menos_est = tab_est["% mismo signo"].idxmin()

# Criterios para el veredicto: DE de la exactitud <= 3 puntos y signo igual en al menos 95 % de las repeticiones
acc_estable = acc_rep.std(ddof=1) <= 0.03
signo_estable = tab_est["% mismo signo"].min() >= 95

In [ ]:
txt_acc = "la exactitud es estable" if acc_estable else "la exactitud varía bastante entre particiones"
txt_signo = ("la dirección de los efectos es estable" if signo_estable else
             f"la dirección de algunos efectos no es estable ({menos_est} cambia de signo en parte de las repeticiones)")
txt_final = ("Por lo tanto, el modelo puede considerarse estable en su capacidad de clasificar y en el sentido de sus "
             "efectos, aunque la magnitud exacta de los coeficientes cambia de una partición a otra por el tamaño "
             "reducido de la muestra (178 vinos)." if (acc_estable and signo_estable) else
             "Por lo tanto, el modelo no puede considerarse completamente estable y sus resultados deben interpretarse "
             "con cautela, dado el tamaño reducido de la muestra (178 vinos).")

md(f"**Interpretación:** Al repetir la partición, la exactitud en prueba tuvo una media de {100 * acc_rep.mean():.1f} % y "
   f"una desviación estándar de {100 * acc_rep.std(ddof=1):.1f} puntos; el 95 % central de los valores está entre "
   f"{100 * np.quantile(acc_rep, 0.025):.1f} % y {100 * np.quantile(acc_rep, 0.975):.1f} %, con extremos de "
   f"{100 * acc_rep.min():.1f} % y {100 * acc_rep.max():.1f} %. La exactitud de {100 * exactitud_m:.1f} % obtenida con la "
   f"partición inicial es, por lo tanto, una de las muchas posibles, y lo razonable es reportar el rango. En cuanto a los "
   f"coeficientes, el signo se mantuvo en al menos el {tab_est['% mismo signo'].min():.1f} % de las repeticiones en todos "
   f"los casos; el coeficiente menos estable fue {menos_est}. Los intervalos de las razones de riesgo relativo que "
   f"incluyen el valor 1 (sin efecto) corresponden a: {texto_inc}.\n\n"
   f"**En resumen:** {txt_acc} (desviación estándar de {100 * acc_rep.std(ddof=1):.1f} puntos; se considera estable si es "
   f"de 3 puntos o menos) y {txt_signo} (el signo se mantiene en al menos 95 % de las repeticiones en cada coeficiente). "
   f"{txt_final}")

## 2.6. Conclusión del ejercicio 2

In [ ]:
md(f"Con tres variables químicas (alcohol, ácido málico e intensidad del color) el modelo multinomial distingue los tres "
   f"tipos de vino con una exactitud de {100 * exactitud_m:.1f} % en datos de prueba. La intensidad del color y el "
   f"contenido de alcohol son las características que más ayudan a diferenciar las clases frente a la clase 1 (ver los "
   f"p-valores de la tabla de coeficientes), y el ácido málico aporta sobre todo para identificar la clase 3. Como "
   f"limitaciones, la muestra es pequeña (178 vinos) y los predictores se usaron en su escala original. Al repetir 200 "
   f"veces la partición, la exactitud promedio fue de {100 * acc_rep.mean():.1f} % (desviación estándar de "
   f"{100 * acc_rep.std(ddof=1):.1f} puntos), por lo que conviene reportar la exactitud como un rango y no como un valor "
   f"único.")

# Comparación final de los modelos

| Aspecto | Logístico binario | Logístico multinomial |
|:---|:---|:---|
| Número de categorías de respuesta | 2 (abandona / no abandona) | 3 (clases de vino 1, 2 y 3) |
| Categoría de referencia | No abandona (0); se modela la probabilidad del evento 1 | Clase 1; las clases 2 y 3 se comparan contra ella |
| Interpretación de coeficientes | Un solo conjunto de coeficientes; exp(coef) es el OR del abandono frente a la permanencia | Un conjunto de coeficientes por cada clase frente a la referencia; exp(coef) es la razón de momios de esa clase frente a la clase 1 |
| Tipo de probabilidades estimadas | Una probabilidad, P(abandono), con P(no abandono) = 1 - P | Un vector de tres probabilidades que suma 1; se clasifica en la de mayor valor |

**Reflexión final:** El modelo binario se usa cuando la respuesta tiene dos categorías, como abandona o no abandona; el multinomial, cuando hay tres o más categorías nominales sin orden, como los tipos de vino. En ambos casos, interpretar las probabilidades y los odds ratios describe la **asociación** entre predictores y respuesta en los datos con que se ajustó el modelo, mientras que comprobar el desempeño con datos nuevos mide si el modelo **generaliza**. Un modelo puede ajustar bien la muestra de entrenamiento y fallar con observaciones nuevas, por eso se reservaron datos de prueba para la matriz de confusión y la exactitud. Además, en el caso binario la exactitud sola puede ser engañosa si una clase es minoritaria.